# Data Mining Assignment: Apache Spark Data Lake Preprocessing
**Environment**: Google Colab + PySpark  
**Dataset**: `sales.csv`  
**Student Name**: Rahul Bharadwaz  
**Roll Number**: LCI2023019  
**Architecture**: Two-layer Medallion Data Lake (Bronze & Silver layers)

---
### Assignment Objective
Using the provided `sales.csv` dataset, create a simple Data Lake structure in Google Colab. Keep the raw data unchanged in the **Bronze** layer, perform basic preprocessing with **Apache Spark**, and save the processed clean data in the **Silver** layer. No Gold layer or detailed analytics is required.


## 1. Data Lake Structure Setup
In the Colab runtime, we create the required folder hierarchy:
```text
data_lake/
|-- bronze/
|   `-- sales.csv
`-- silver/
    `-- preprocessed_sales/
```
The Bronze file must remain exactly as provided without any modifications.


In [1]:
# Step 1.1: Install PySpark in Google Colab environment
!pip install -q pyspark

# Step 1.2: Create Data Lake folder structure
import os
import shutil
import urllib.request

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Step 1.3: Place raw sales.csv into the Bronze layer (unmodified)
dataset_url = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"
bronze_csv_path = "data_lake/bronze/sales.csv"

if not os.path.exists(bronze_csv_path):
    print(f"Downloading raw dataset to Bronze layer: {bronze_csv_path}...")
    urllib.request.urlretrieve(dataset_url, bronze_csv_path)
    print("Bronze dataset successfully placed.")
else:
    print(f"Bronze dataset already present at {bronze_csv_path}.")

# Display the created directory tree
print("
Data Lake directory structure:")
for root, dirs, files in os.walk("data_lake"):
    level = root.replace("data_lake", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")


Bronze dataset already present at data_lake/bronze/sales.csv.

Data Lake directory structure:
data_lake/
    bronze/
        sales.csv
    silver/
        preprocessed_sales/


## 2. Load and Inspect with Spark
In this section, we:
1. Initialize a `SparkSession`.
2. Ingest `sales.csv` from the **Bronze** layer.
3. Display the schema, inspect sample rows, and compute the total record count.
4. Dynamically identify all discrepancies using **Spark operations** (no hardcoded counts):
   - Duplicate rows
   - Null / missing values
   - Text formatting and capitalization inconsistencies
   - Invalid numerical values (`quantity`, `unit_price`, `discount_percent`)
   - Inconsistent and invalid dates (`order_date`)


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnan, isnull, trim, initcap,
    coalesce, to_date, date_format, rlike
)
from pyspark.sql.types import IntegerType, DoubleType, StringType

# Initialize SparkSession with legacy time parser policy for robust date handling
spark = SparkSession.builder \
    .appName("DataLake_Bronze_to_Silver_Preprocessing") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

# Read raw dataset from Bronze layer (strictly keeping raw data unchanged)
bronze_df = spark.read.csv(
    "data_lake/bronze/sales.csv",
    header=True,
    inferSchema=True
)

print("Bronze Dataset Schema:")
bronze_df.printSchema()

bronze_total_records = bronze_df.count()
print(f"Total Record Count in Bronze Layer: {bronze_total_records}")

print("\nSample 5 Rows from Bronze Layer:")
bronze_df.show(5, truncate=False)


Bronze Dataset Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Total Record Count in Bronze Layer: 1000

Sample 5 Rows from Bronze Layer:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--

In [3]:
print("=================================================================")
print("DISCREPANCY IDENTIFICATION USING SPARK OPERATIONS")
print("=================================================================")

# 1. Exact Duplicate Rows
bronze_count = bronze_df.count()
bronze_distinct_count = bronze_df.distinct().count()
duplicate_rows_count = bronze_count - bronze_distinct_count
print(f"1. DUPLICATE CHECK:")
print(f"   Total rows: {bronze_count}")
print(f"   Distinct rows: {bronze_distinct_count}")
print(f"   Exact duplicate rows detected: {duplicate_rows_count}")

# 2. Null / Missing Values per Column
print(f"\n2. NULL / MISSING VALUES COUNT PER COLUMN:")
null_counts_df = bronze_df.select([
    count(when(isnull(c) | (trim(col(c)) == ""), c)).alias(c)
    for c in bronze_df.columns
])
null_counts_df.show()

# 3. Numerical Outliers & Invalids
print("3. NUMERICAL BOUNDS VALIDATION:")
invalid_qty_count = bronze_df.filter(col("quantity") <= 0).count()
invalid_price_count = bronze_df.filter(col("unit_price") <= 0).count()
invalid_discount_count = bronze_df.filter((col("discount_percent") < 0) | (col("discount_percent") > 100)).count()

print(f"   Rows with quantity <= 0: {invalid_qty_count}")
print(f"   Rows with unit_price <= 0: {invalid_price_count}")
print(f"   Rows with discount_percent < 0 or > 100: {invalid_discount_count}")

print("\nSample Numerical Anomalies:")
bronze_df.filter(
    (col("quantity") <= 0) | (col("unit_price") <= 0) |
    (col("discount_percent") < 0) | (col("discount_percent") > 100)
).select("order_id", "product", "quantity", "unit_price", "discount_percent", "order_status").show(5)

# 4. Text-format & Capitalization Inconsistencies
print("4. TEXT FORMATTING & CAPITALIZATION INCONSISTENCIES:")
print("   Distinct Category values before standardization:")
bronze_df.select("category").distinct().show()

print("   Distinct City values with mixed casing / whitespace (sample):")
bronze_df.select("city").distinct().show(8)

# 5. Inconsistent / Invalid Dates
print("5. DATE INCONSISTENCY & INVALID DATES CHECK:")
# Parse using multi-pattern coalesce
parsed_dates_test = coalesce(
    to_date(col("order_date"), "yyyy-MM-dd"),
    to_date(col("order_date"), "yyyy/MM/dd"),
    to_date(col("order_date"), "MM-dd-yyyy")
)

invalid_dates_df = bronze_df.filter(parsed_dates_test.isNull())
invalid_dates_count = invalid_dates_df.count()
print(f"   Rows with completely invalid / unparseable dates: {invalid_dates_count}")
invalid_dates_df.select("order_id", "customer_id", "order_date").show(truncate=False)

alternate_format_dates_df = bronze_df.filter(
    parsed_dates_test.isNotNull() & ~col("order_date").rlike("^\\d{4}-\\d{2}-\\d{2}$")
)
print(f"   Rows with valid dates in non-standard formats (e.g. yyyy/MM/dd, MM-dd-yyyy): {alternate_format_dates_df.count()}")
alternate_format_dates_df.select("order_id", "order_date").show()


DISCREPANCY IDENTIFICATION USING SPARK OPERATIONS
1. DUPLICATE CHECK:
   Total rows: 1000
   Distinct rows: 990
   Exact duplicate rows detected: 10

2. NULL / MISSING VALUES COUNT PER COLUMN:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

3. NUMERICAL BOUNDS VALIDATION:
   Rows with quantity <= 0: 12
   Rows with unit_price <= 0: 8
   Row

## 3. Basic Preprocessing - Silver Layer
We execute the required preprocessing transformations in Apache Spark following the 6 assignment rules:

1. **Remove exact duplicate rows**: Using `dropDuplicates()`.
2. **Handle missing values reasonably**: Impute missing non-critical customer metadata (`customer_name`, `payment_method`, `city`) with `'Unknown'` to preserve valid commercial sales records.
3. **Trim extra spaces and standardize capitalization**: Apply `trim()` across all string columns and `initcap()` on categorical text fields (`category`, `city`, `state`, `payment_method`, `order_status`).
4. **Apply basic numerical validation**: Filter out invalid transactions failing physical constraints (`quantity > 0`, `unit_price > 0`, `0 <= discount_percent <= 100`).
5. **Convert valid dates to one consistent format & handle invalid dates**: Multi-pattern parsing (`yyyy-MM-dd`, `yyyy/MM/dd`, `MM-dd-yyyy`) into standard `yyyy-MM-dd`, discarding records with unrecoverable invalid dates (`'not-a-date'`, `'31/02/2026'`, etc.).
6. **Preserve useful records**: Ensure valid transactions with minor metadata gaps are retained rather than aggressively purged.

Save the resulting DataFrame into the **Silver** folder.


In [4]:
# 1. Remove exact duplicate rows
step1_df = bronze_df.dropDuplicates()
print(f"1. Rows after dropDuplicates: {step1_df.count()} (dropped {bronze_df.count() - step1_df.count()} duplicate rows)")

# 2. Handle missing values using a reasonable method
# Preserving transactions by imputing missing optional metadata with 'Unknown'
step2_df = step1_df.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})
print("2. Missing values handled via imputation with 'Unknown'.")

# 3. Trim extra spaces and standardize obvious capitalization inconsistencies
string_cols = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
step3_df = step2_df
for c in string_cols:
    step3_df = step3_df.withColumn(c, trim(col(c)))

casing_cols = ["category", "payment_method", "city", "state", "order_status"]
for c in casing_cols:
    step3_df = step3_df.withColumn(c, initcap(col(c)))
print("3. Extra spaces trimmed and capitalization standardized with initcap().")

# 4. Apply basic validation to quantity, unit price, and discount percentage
# Retain records where quantity > 0, unit_price > 0, and 0 <= discount_percent <= 100
step4_df = step3_df.filter(
    (col("quantity") > 0) &
    (col("unit_price") > 0) &
    (col("discount_percent") >= 0) &
    (col("discount_percent") <= 100)
)
print(f"4. Rows after numerical validation: {step4_df.count()}")

# 5. Convert valid dates to one consistent format (yyyy-MM-dd) and handle clearly invalid dates
parsed_order_date = coalesce(
    to_date(col("order_date"), "yyyy-MM-dd"),
    to_date(col("order_date"), "yyyy/MM/dd"),
    to_date(col("order_date"), "MM-dd-yyyy")
)

# Parse valid dates to standard format; filter out clearly invalid date records (which parse to NULL)
silver_df = step4_df.withColumn("parsed_date", parsed_order_date) \
                    .filter(col("parsed_date").isNotNull()) \
                    .withColumn("order_date", date_format(col("parsed_date"), "yyyy-MM-dd")) \
                    .drop("parsed_date")

print(f"5. Rows after date parsing and invalid date removal: {silver_df.count()}")
print(f"6. Useful records successfully preserved in Silver DataFrame.")


1. Rows after dropDuplicates: 990 (dropped 10 duplicate rows)
2. Missing values in customer_name, payment_method, city imputed with 'Unknown'.
3. Extra spaces trimmed and capitalization standardized with initcap().
4. Rows after numerical validation: 962
5. Rows after date parsing and invalid date removal: 958
6. Useful records successfully preserved in Silver DataFrame.


### Save Silver Layer Output
We save the preprocessed DataFrame into the Data Lake `silver/preprocessed_sales/` directory and also export `sales_silver.csv` into the required `silver/` folder for assignment submission.


In [5]:
# Save DataFrame inside the Data Lake Silver folder (CSV format)
silver_lake_output_path = "data_lake/silver/preprocessed_sales"
silver_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(silver_lake_output_path)
print(f"Preprocessed DataFrame saved inside Silver folder: {silver_lake_output_path}")

# Export sales_silver.csv for GitHub submission folder
os.makedirs("silver", exist_ok=True)
submission_csv_path = "silver/sales_silver.csv"

# Convert to pandas and export single clean CSV file
silver_pandas_df = silver_df.toPandas()
silver_pandas_df.to_csv(submission_csv_path, index=False)
print(f"Generated submission file: {submission_csv_path} with {len(silver_pandas_df)} records.")


Preprocessed DataFrame saved inside Silver folder: data_lake/silver/preprocessed_sales
Generated submission file: silver/sales_silver.csv with 958 records.


## 4. Verification
Per the assignment requirements:
1. Show the row count before and after preprocessing.
2. Read the Silver data back with Spark and display a few rows.
3. State which preprocessing operations were performed.


In [6]:
print("=================================================================")
print("VERIFICATION OF PREPROCESSING RESULTS")
print("=================================================================")

# 1. Row count before and after preprocessing
count_before = bronze_df.count()
count_after = silver_df.count()
records_removed = count_before - count_after

print(f"Row count BEFORE preprocessing (Bronze layer): {count_before}")
print(f"Row count AFTER preprocessing  (Silver layer): {count_after}")
print(f"Total invalid / duplicate rows removed       : {records_removed}")

# 2. Read the Silver data back with Spark and display sample rows
print("\nReading Silver data back with Apache Spark:")
silver_readback_df = spark.read.csv(silver_lake_output_path, header=True, inferSchema=True)

print("Schema of Silver Dataset:")
silver_readback_df.printSchema()

print(f"Total records read back from Silver layer: {silver_readback_df.count()}")
print("\nSample 10 Rows from Silver Layer:")
silver_readback_df.show(10, truncate=False)

# Verification checks confirming zero remaining discrepancies
print("\nVerification Sanity Checks:")
print(f"- Remaining duplicate rows: {silver_readback_df.count() - silver_readback_df.distinct().count()}")
print(f"- Remaining null customer_name: {silver_readback_df.filter(isnull(col('customer_name'))).count()}")
print(f"- Remaining quantity <= 0: {silver_readback_df.filter(col('quantity') <= 0).count()}")
print(f"- Remaining unit_price <= 0: {silver_readback_df.filter(col('unit_price') <= 0).count()}")
print(f"- Remaining discount_percent out of range [0, 100]: {silver_readback_df.filter((col('discount_percent') < 0) | (col('discount_percent') > 100)).count()}")
print(f"- Remaining non-standard dates: {silver_readback_df.filter(~col('order_date').rlike('^\\d{4}-\\d{2}-\\d{2}$')).count()}")


VERIFICATION OF PREPROCESSING RESULTS
Row count BEFORE preprocessing (Bronze layer): 1000
Row count AFTER preprocessing  (Silver layer): 958
Total invalid / duplicate rows removed       : 42

Reading Silver data back with Apache Spark:
Schema of Silver Dataset:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Total records read back from Silver layer: 958

Sample 10 Rows from Silver Layer:
+--------+-----------+--------------+--------------+---------------+--------+----------+----------------

### 3. Summary of Preprocessing Operations Performed
| # | Step / Requirement | Preprocessing Operation Performed | Impact / Justification |
| :--- | :--- | :--- | :--- |
| **1** | **Data Lake Architecture** | Created `data_lake/bronze/` and `data_lake/silver/preprocessed_sales/`. | Raw `sales.csv` kept strictly untouched in Bronze; clean data stored in Silver. |
| **2** | **Duplicate Elimination** | Executed `bronze_df.dropDuplicates()`. | Removed 10 exact duplicate records, eliminating redundant transaction entries. |
| **3** | **Missing Value Handling** | Imputed missing `customer_name`, `payment_method`, and `city` with `'Unknown'`. | Preserves commercial transaction integrity without dropping valid orders. |
| **4** | **Whitespace & Casing** | Applied `trim()` across all string columns; applied `initcap()` to categories and locations. | Resolved erratic spacing (`' Mumbai '`) and normalized capitalization (`'CLOTHING'` $\rightarrow$ `'Clothing'`). |
| **5** | **Numerical Validation** | Filtered `quantity > 0`, `unit_price > 0`, and `0 <= discount_percent <= 100`. | Eliminated physically impossible negative prices, 0-quantity orders, and invalid discounts. |
| **6** | **Date Normalization** | Coalesced formats (`yyyy-MM-dd`, `yyyy/MM/dd`, `MM-dd-yyyy`) to standard `yyyy-MM-dd`; dropped unparseable dates. | Retained valid non-standard dates while eliminating corrupted strings (`'not-a-date'`) and impossible dates (`'31/02/2026'`). |
| **7** | **Record Preservation** | Retained useful records with minor metadata gaps. | Balanced data cleanliness with maximum information retention. |
